In [1]:
# 1. Image Classifier CNN

In [3]:
import tensorflow as tf
import keras

In [4]:
(X_train, y_train) , (X_test, y_test) = keras.datasets.cifar10.load_data()

170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 1600s 9us/step


In [5]:
X_train = X_train.astype("float32") / 255.0
X_test = X_test.astype("float32") / 255.0

In [6]:
model = keras.models.Sequential([
    keras.layers.Conv2D(32, (3,3), padding="same", activation="relu", input_shape=(32,32,3)),
    keras.layers.MaxPool2D((2,2)),

    keras.layers.Conv2D(64, (3,3), padding="same", activation="relu"),
    keras.layers.MaxPool2D((2,2)),

    keras.layers.Flatten(),
    keras.layers.Dense(128, activation="relu"),
    keras.layers.Dense(10, activation="softmax")
])

/usr/local/lib/python3.13/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [7]:
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [8]:
model.fit(
    X_train,
    y_train,
    epochs=10,
    batch_size=64,
    validation_data=(X_test, y_test)
)

Epoch 1/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 10s 8ms/step - accuracy: 0.5085 - loss: 1.3854 - val_accuracy: 0.6063 - val_loss: 1.1231
Epoch 2/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.6377 - loss: 1.0346 - val_accuracy: 0.6451 - val_loss: 1.0145
Epoch 3/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.6875 - loss: 0.8974 - val_accuracy: 0.6754 - val_loss: 0.9332
Epoch 4/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.7193 - loss: 0.8043 - val_accuracy: 0.6899 - val_loss: 0.8961
Epoch 5/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.7511 - loss: 0.7147 - val_accuracy: 0.6996 - val_loss: 0.8727
Epoch 6/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.7779 - loss: 0.6366 - val_accuracy: 0.7009 - val_loss: 0.8891
Epoch 7/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.8052 - loss: 0.5630 - val_accuracy: 0.7108 - val_loss: 0.8899
Epoch 8/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.8292 - loss: 0.4939 - val_accuracy: 0

In [9]:
loss, accuracy = model.evaluate(X_test, y_test)
print("Test Accuracy:",accuracy)

313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7132 - loss: 0.9777
Test Accuracy: 0.7131999731063843


In [11]:
# 2. Transfer Learning

In [12]:
(x_train,y_train) , (x_test, y_test) = keras.datasets.fashion_mnist.load_data()

29515/29515 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
26421880/26421880 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
5148/5148 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
4422102/4422102 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [13]:
BATCH_SIZE = 32
IMAGE_SIZE = 64

train_ds = tf.data.Dataset.from_tensor_slices((x_train,y_train))
test_ds = tf.data.Dataset.from_tensor_slices((x_test,y_test))

In [17]:
def preprocess(image, label):
  image = tf.expand_dims(image, axis=-1)

  image = tf.image.grayscale_to_rgb(image)

  image = tf.image.resize(
      image, (IMAGE_SIZE,IMAGE_SIZE)
  )

  image = tf.cast(image, tf.float32) / 255.0

  return image, label

In [19]:
train_ds = train_ds.map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
test_ds = test_ds.map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)

train_ds = train_ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
test_ds = train_ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

In [20]:
from keras.applications import ResNet50
base_model = ResNet50(
    weights = "imagenet",
    include_top = False,
    input_shape=(IMAGE_SIZE,IMAGE_SIZE,3)
)
base_model.trainable = False

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [ ]:
model = keras.models.Sequential([
    base_model,
    keras.layers.GlobalAveragePooling2D()
    keras.layers.Dense(128, activation="relu"),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(10, activation="softmax")
])

In [ ]:
model.compile(
    optimizer = "adam",
    loss = "sparse_categorical_crossentropy",
    metrics = ["accuracy"]
)

In [ ]:
history = model.fit(
    train_ds,
    epochs=3,
    validation_data = test_ds
)

In [ ]:
  # 2. Fine Tuning

In [ ]:
(x_train,y_train), (x_test,y_test) = keras.datasets.fashion_mnist.load_data()

train_ds = tf.data.Dataset.from_tensor_slices((x_train,y_train))
test_ds = tf.data.Dataset.from_tensor_slices((x_test,y_test))

In [ ]:
IMG_SIZE = 64
BATCH_SIZE = 32

In [ ]:
def preprocess(image, label):
  image = tf.expand_dims(image, axis=-1)
  image = tf.image.grayscale_to_rgb(image)
  image = tf.resize(
      image,(IMG_SIZE,IMG_SIZE)
  )
  image = tf.cast(image, tf.float32) / 255.0
  return image, label

In [ ]:
train_ds = train_ds.map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
test_ds = test_ds.map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)

train_ds = train_ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
test_ds = test_ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

In [ ]:
from keras.applications import VGG16
base_model = VGG16(
    weights = "imagenet",
    input_shape = (IMG_SIZE,IMG_SIZE,3),
    include_top = False
)

In [ ]:
model = keras.models.Sequential([
    base_model,
    keras.layers.GlobalAveragePooling2D(),
    keras.layers.Dense(128, activation="relu"),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(10, activation="softmax")
])

In [ ]:
model.compile(
    optimizer = keras.optimizers.AdamW(learning_rate=0.0001),
    loss = "sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
history = model.fit(
  train_ds,
  validation_data=test_data,
  epochs=3
)

In [ ]:
base_model.trainable = True

for layer in base_model.layers[:-4]:
  layer.trainable=False

print("\nTrainable layers:",sum(layer.trainable for layer in base_model.layers))


model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.00001
    ),

    loss="sparse_categorical_crossentropy",

    metrics=["accuracy"]
)


print("\nPHASE 2: Fine-tuning ResNet")

history2 = model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=3
)


test_loss, test_accuracy = model.evaluate(test_ds)

print("\nFinal Test Accuracy:", test_accuracy)